In [ ]:
!unzip /content/screenshots.zip -d /content/screenshots

Archive:  /content/screenshots.zip
   creating: /content/screenshots/screenshots/
  inflating: /content/screenshots/screenshots/60061.jpg  
  inflating: /content/screenshots/screenshots/58419.jpg  
  inflating: /content/screenshots/screenshots/42214.jpg  
  inflating: /content/screenshots/screenshots/47883.jpg  
  inflating: /content/screenshots/screenshots/70502.jpg  
  inflating: /content/screenshots/screenshots/10798.jpg  
  inflating: /content/screenshots/screenshots/66870.jpg  
  inflating: /content/screenshots/screenshots/14247.jpg  
  inflating: /content/screenshots/screenshots/11150.jpg  
  inflating: /content/screenshots/screenshots/28535.jpg  
  inflating: /content/screenshots/screenshots/14554.jpg  
  inflating: /content/screenshots/screenshots/8869.jpg  
  inflating: /content/screenshots/screenshots/58270.jpg  
  inflating: /content/screenshots/screenshots/26425.jpg  
  inflating: /content/screenshots/screenshots/1996.jpg  
  inflating: /content/screenshots/screenshots/6666

In [ ]:
!pip install -q torch torchvision pillow transformers faiss-cpu
!pip install -q "langchain>=0.2.0" "langchain-community>=0.2.0"
!pip install -q json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
# ============================================
# CELL 2 — Imports
# ============================================
import os, csv, json, re
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch

from transformers import (
    CLIPProcessor,
    CLIPModel,
    AutoProcessor,
    AutoModelForVision2Seq,
)

import faiss

from langchain_community.vectorstores import FAISS
from langchain_community.docstore import InMemoryDocstore
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings


In [ ]:
# ============================================
# CELL 3 — Paths (adjust if needed)
# ============================================
screenshot_dir   = "/content/screenshots/screenshots"
topic_csv_path   = "/content/design_topics.csv"
save_dir         = "ss_faiss_langchain"

# NEW: where description .txt files will be stored
desc_txt_dir     = "/content/descriptions.txt"

# NEW: where generated test case CSVs will be stored
tc_csv_dir       = "/content/testcases.csv"

os.makedirs(desc_txt_dir, exist_ok=True)
os.makedirs(tc_csv_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
# ============================================
# CELL 4 — Load models (CLIP + Qwen2.5-VL)
# ============================================
clip_name = "laion/CLIP-ViT-L-14-laion2B-s32B-b82K"
clip_model = CLIPModel.from_pretrained(clip_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_name)

MAX_PROMPT_CHARS = 2200
MAX_NEW_TOKENS   = 700

vlm_name = "Qwen/Qwen2.5-VL-7B-Instruct"
vlm_processor = AutoProcessor.from_pretrained(vlm_name, trust_remote_code=True)
vlm_model = AutoModelForVision2Seq.from_pretrained(
    vlm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
).to(device)

print("Loaded CLIP + Qwen2.5-VL-7B-Instruct.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loaded CLIP + Qwen2.5-VL-7B-Instruct.


In [ ]:
# ============================================
# CELL 5 — Safe image loading
# ============================================
def load_image_safe(path):
    try:
        img = Image.open(path)
    except UnidentifiedImageError:
        print(f"[SKIP] Unreadable image: {path}")
        return None
    except Exception as e:
        print(f"[SKIP] Error opening {path}: {e}")
        return None

    if img.mode == "P":
        img = img.convert("RGBA")
    if img.mode in ("RGBA", "LA"):
        img = img.convert("RGB")
    else:
        img = img.convert("RGB")
    return img


def collect_valid_images(root):
    valid, bad = [], []
    for fname in sorted(os.listdir(root)):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        path = os.path.join(root, fname)
        img = load_image_safe(path)
        if img is None:
            bad.append(path)
        else:
            valid.append(path)
    return valid, bad


In [ ]:
# ============================================
# CELL 6 — CLIP embedding helpers
# ============================================
@torch.no_grad()
def embed_pil_image(img: Image.Image):
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    emb = clip_model.get_image_features(**inputs)
    emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    return emb.squeeze().detach().cpu().numpy().astype("float32")


@torch.no_grad()
def embed_image(path):
    img = load_image_safe(path)
    if img is None:
        return None
    return embed_pil_image(img)

In [ ]:
# ============================================
# CELL 7 — Qwen output cleanup + VLM generation (image + text)
# ============================================
def _strip_qwen_chat_header(text: str) -> str:
    marker = "assistant\n"
    last_idx = text.rfind(marker)
    if last_idx != -1:
        return text[last_idx + len(marker):].strip()
    return text.strip()


@torch.no_grad()
def generate_with_vlm(img: Image.Image, prompt_text: str, max_new_tokens=MAX_NEW_TOKENS):
    prompt_text = prompt_text[:MAX_PROMPT_CHARS]

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    chat_text = vlm_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = vlm_processor(
        text=[chat_text],
        images=[img],
        return_tensors="pt",
    ).to(device)

    generated_ids = vlm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=3,
        repetition_penalty=1.08,
    )

    raw = vlm_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0]

    return _strip_qwen_chat_header(raw)


In [ ]:
# ============================================
# CELL 8 — TEXT-ONLY generation (Stage 2 uses this)
# ============================================
@torch.no_grad()
def generate_text_only(prompt_text: str, max_new_tokens=900):
    prompt_text = prompt_text[:MAX_PROMPT_CHARS]

    messages = [
        {"role": "user", "content": [{"type": "text", "text": prompt_text}]}
    ]

    chat_text = vlm_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = vlm_processor(
        text=[chat_text],
        return_tensors="pt",
    ).to(device)

    generated_ids = vlm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=3,
        repetition_penalty=1.08,
    )

    raw = vlm_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return _strip_qwen_chat_header(raw)


In [ ]:
# ============================================
# CELL 9 — Crop status bar (unchanged)
# ============================================
def crop_status_bar(img: Image.Image, top_ratio=0.12):
    w, h = img.size
    top_h = max(40, int(h * top_ratio))
    return img.crop((0, 0, w, top_h))

In [ ]:
# ============================================
# CELL 10 — Prompts (Stage 1)
# ============================================
STATUSBAR_PROMPT = """
You are a senior mobile UI analyst.

Focus ONLY on the STATUS BAR (the thin strip at the very top of the screen).
Ignore everything below it.

Your task is to report what is visually present, with confidence awareness.

Rules:
- If a symbol is clearly recognizable by its visual shape or logo, name it.
- If a symbol looks like a known system icon but is not fully certain, write:
  "likely <icon name> (low confidence)".
- If a symbol cannot be reliably identified, write:
  "unknown icon" with a brief shape description.
- Do NOT assume icons just because they are common.
- Do NOT infer icons from context.
- Time should be reported if readable.

Scan strictly from LEFT → RIGHT.

Return EXACTLY two sections:

1) Left-side notification/app icons:
   - List icons in order.
   - Use app names only when logos are unmistakable.

2) Right-side system/status icons + time:
   - List icons in order.
   - Use confidence labels where needed.
   - End with time if readable.

Be literal, visual, and conservative.
"""

SCREENSHOT_PROMPT = """
You are a senior mobile UI and UX analyst.

You are looking at ONE FINAL IMPLEMENTED MOBILE APP SCREENSHOT (a fully designed UI).
Describe the screen in detail for a developer who cannot see the image.

ABSOLUTE RULES:
- Always talk in terms of UI components (status bar, header/app bar, title, subtitle, image, list row, toggle, chevron, button, footer, bottom nav).
- If any text is readable (titles, labels, button text), include it verbatim.
- Do NOT skip the bottom area: capture CTA buttons and any footer/bottom nav if present.

OUTPUT STRUCTURE (use these headings exactly):
A) Screen purpose and type:
B) Layout top → bottom:
C) Component inventory (grouped):
D) Primary actions + navigation:
Write 14–22 sentences total (compact, but complete).
End with a bullet recap:
- Screen type/purpose
- Key components
- Primary CTA(s)
- Navigation/footer
"""


In [ ]:
# ============================================
# CELL 11 — Caches
# ============================================
_screenshot_desc_cache = {}
_statusbar_cache = {}


In [ ]:
# ============================================
# CELL 12 — Load topics CSV: screen_id -> topic
# ============================================
id_to_topic = {}
with open(topic_csv_path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    for row in reader:
        if not row:
            continue
        sid   = row[0].strip()
        topic = row[1].strip() if len(row) > 1 else "unknown"
        id_to_topic[sid] = topic

print("Total IDs with topics:", len(id_to_topic))


Total IDs with topics: 1460


In [ ]:
# ============================================
# CELL 13 — Collect screenshots + mapping
# ============================================
valid_screenshots, bad_screenshots = collect_valid_images(screenshot_dir)
print(f"Valid screenshots:      {len(valid_screenshots)}")
print(f"Unreadable screenshots: {len(bad_screenshots)}")

screenshot_by_id = {}
for p in valid_screenshots:
    pid = os.path.splitext(os.path.basename(p))[0]
    screenshot_by_id[pid] = p

def get_id_from_path(path):
    return os.path.splitext(os.path.basename(path))[0]

print("Example screenshot_by_id keys:", list(screenshot_by_id.keys())[:5])


Valid screenshots:      1460
Unreadable screenshots: 0
Example screenshot_by_id keys: ['10128', '10180', '10183', '10208', '10214']


In [ ]:
# ============================================
# CELL 14 — Stage 1: Describe screenshot (UPDATED: also saves .txt)
# ============================================
def save_description_txt(screen_id: str, merged_description: str, out_dir: str = desc_txt_dir):
    out_path = os.path.join(out_dir, f"{screen_id}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(merged_description.strip() + "\n")
    return out_path


def describe_screenshot(path, save_txt=True):
    if path in _screenshot_desc_cache:
        return _screenshot_desc_cache[path]

    img = load_image_safe(path)
    if img is None:
        merged = "Screenshot could not be opened."
        _screenshot_desc_cache[path] = merged
        return merged

    full_desc = generate_with_vlm(img, SCREENSHOT_PROMPT, max_new_tokens=MAX_NEW_TOKENS)

    sb_img = crop_status_bar(img, top_ratio=0.12)
    sb_desc = generate_with_vlm(sb_img, STATUSBAR_PROMPT, max_new_tokens=220)
    _statusbar_cache[path] = sb_desc

    merged = (
        full_desc.strip()
        + "\n\n"
        + "STATUS BAR (cropped pass):\n"
        + sb_desc.strip()
    )

    _screenshot_desc_cache[path] = merged

    if save_txt:
        sid = get_id_from_path(path)
        save_description_txt(sid, merged, out_dir=desc_txt_dir)

    return merged


In [ ]:
# ============================================
# CELL 15 — Build + Save + Reload screenshot vector store (unchanged workflow)
# ============================================
def build_and_save_screenshot_vectorstore(paths, save_dir=save_dir):
    ss_vectors = []
    ss_meta = []

    for path in paths:
        sid = os.path.splitext(os.path.basename(path))[0]
        vec = embed_image(path)
        if vec is None:
            continue

        topic = id_to_topic.get(sid, "unknown")
        meta = {
            "screen_id": sid,
            "topic": topic,
            "path": path,
        }

        ss_vectors.append(vec)
        ss_meta.append(meta)

    if len(ss_vectors) == 0:
        raise RuntimeError("No valid screenshot embeddings were created.")

    ss_vectors = np.stack(ss_vectors).astype("float32")
    d = ss_vectors.shape[1]

    faiss_index = faiss.IndexFlatIP(d)
    docstore    = InMemoryDocstore({})
    id_map      = {}

    ss_vs_build = FAISS(
        embedding_function=None,
        index=faiss_index,
        docstore=docstore,
        index_to_docstore_id=id_map,
    )

    ss_docs = [Document(page_content="screenshot", metadata=m) for m in ss_meta]
    ids = [str(i) for i in range(len(ss_docs))]
    text_embeddings = [(ss_docs[i].page_content, ss_vectors[i]) for i in range(len(ss_docs))]
    metadatas       = [ss_docs[i].metadata for i in range(len(ss_docs))]

    ss_vs_build.add_embeddings(text_embeddings, metadatas, ids)
    ss_vs_build.save_local(save_dir)

    print(f"Saved screenshot FAISS index to: {save_dir}/")
    print("Index ntotal:", ss_vs_build.index.ntotal)

    return ss_vectors, ss_meta


class DummyEmbeddings(Embeddings):
    def embed_documents(self, texts):
        return [[0.0] * 1 for _ in texts]
    def embed_query(self, text):
        return [0.0] * 1


def load_screenshot_vectorstore(save_dir=save_dir):
    ss_vs = FAISS.load_local(
        save_dir,
        embeddings=DummyEmbeddings(),
        allow_dangerous_deserialization=True,
    )
    print("Reloaded screenshot vector store. ntotal:", ss_vs.index.ntotal)

    ss_meta = []
    ss_vectors = []

    for i in range(ss_vs.index.ntotal):
        doc_id = ss_vs.index_to_docstore_id[i]
        doc    = ss_vs.docstore.search(doc_id)
        ss_meta.append(dict(doc.metadata))
        ss_vectors.append(ss_vs.index.reconstruct(i))

    ss_vectors = np.stack(ss_vectors).astype("float32")
    return ss_vs, ss_vectors, ss_meta


ss_vectors, ss_meta = build_and_save_screenshot_vectorstore(valid_screenshots, save_dir=save_dir)
ss_vs, ss_vectors_reloaded, ss_meta_reloaded = load_screenshot_vectorstore(save_dir=save_dir)


Saved screenshot FAISS index to: ss_faiss_langchain/
Index ntotal: 1460
Reloaded screenshot vector store. ntotal: 1460


In [ ]:
# ============================================
# CELL 16 — Query Stage 1: show description + save .txt
# ============================================
def analyze_query_description_only(query_screenshot_path: str):
    query_id = get_id_from_path(query_screenshot_path)
    query_topic = id_to_topic.get(query_id, "unknown")

    query_desc = describe_screenshot(query_screenshot_path, save_txt=True)
    txt_path = os.path.join(desc_txt_dir, f"{query_id}.txt")

    return {
        "query_screenshot": query_screenshot_path,
        "query_id": query_id,
        "query_topic": query_topic,
        "query_screenshot_description": query_desc,
        "saved_txt_path": txt_path,
    }


query_screenshot_path = "/content/screenshots/screenshots/15660.jpg"
result = analyze_query_description_only(query_screenshot_path)

print("\n=== QUERY RESULT ===")
print("Query screenshot:", result["query_screenshot"])
print("Query ID:", result["query_id"])
print("Topic from CSV:", result["query_topic"])
print("Saved TXT:", result["saved_txt_path"])

print("\n--- FULL QUERY SCREENSHOT DESCRIPTION ---\n")
print(result["query_screenshot_description"])



=== QUERY RESULT ===
Query screenshot: /content/screenshots/screenshots/15660.jpg
Query ID: 15660
Topic from CSV: form
Saved TXT: /content/descriptions.txt/15660.txt

--- FULL QUERY SCREENSHOT DESCRIPTION ---

A) Screen purpose and type:
The screen is a form for creating a new contact within a mobile app, likely for business or personal use.

B) Layout top → bottom:
- **Header/App Bar**: Contains a back arrow, the title "New Contact," and a "SAVE" button.
- **Primary Contact Section**: Includes fields for "Saln.," "First Name," "Last Name," "Company Name," and "Contact Display Name."
- **Contact Email Section**: Includes a field for "Contact Email."
- **Contact Information Section**: Includes fields for "Work Phone" and "Mobile."
- **Currency Section**: Displays "Currency" and "USD."
- **Payment Terms Section**: Displays "Payment Terms" and "Due on Receipt."

C) Component inventory (grouped):
- **Header/App Bar**: Back arrow, title "New Contact," "SAVE" button.
- **Form Fields**: "Sal

In [ ]:
# ============================================
# CELL 17 — Stage 2 prompt (test case generation)
# ============================================
TESTCASE_PROMPT_TEMPLATE = """
You are a senior QA engineer (mobile apps). Generate REALISTIC, high-quality test cases from the given screen description.

Input is a SCREEN DESCRIPTION extracted from a mobile screenshot (components + actions + navigation + buttons ).
Your goal: produce a set of test cases that a senior tester would actually execute.

Hard rules:
- Do NOT invent UI elements that are not mentioned.
- Do NOT reference "the image" or "the screenshot"; reference only what is described.
- Prefer deterministic, step-by-step actions.
- Cover: positive flows, negative flows, boundary/validation, toggles state, Clickable buttons, Check boxes, Select, Input, navigation, accessibility/usability, interruption/resume, and data persistence where applicable.
- If a test requires unknown app behavior, write assumption in Preconditions (short).

Return ONLY valid JSON (no markdown, no commentary).

JSON schema:
{{
  "module": "<short module name>",
  "screen_id": "<screen id>",
  "test_cases": [
    {{
      "test_case_id": "TC_<screen_id>_###",
      "title": "<clear title>",
      "priority": "P0|P1|P2",
      "type": "Functional|Negative|UI|Accessibility|Usability|Regression",
      "preconditions": ["..."],
      "test_data": ["..."],
      "steps": ["Step 1 ...", "Step 2 ..."],
      "expected": ["..."]
    }}
  ]
}}

Screen ID: {screen_id}
Topic: {topic}

SCREEN DESCRIPTION:
\"\"\"
{description}
\"\"\"
"""


In [ ]:
from json_repair import repair_json

def _extract_json(text: str) -> str | None:
    """
    Extract the first top-level JSON object from model output.
    Handles extra text before/after JSON.
    """
    if not text:
        return None

    text = text.strip()

    # Fast path
    if text.startswith("{") and text.endswith("}"):
        return text

    # Balanced-brace extraction (robust)
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i+1]

    return None


def parse_testcases_json(raw_text: str):
    """
    1) Extract JSON object
    2) Try json.loads
    3) If fails, attempt json_repair
    4) Try again
    """
    js = _extract_json(raw_text)
    if js is None:
        raise ValueError("Could not locate JSON in model output.")

    # Try normal loads
    try:
        return json.loads(js)
    except json.JSONDecodeError:
        pass

    # Common cleanup: remove trailing commas
    js2 = re.sub(r",\s*([}\]])", r"\1", js)

    try:
        return json.loads(js2)
    except json.JSONDecodeError:
        pass

    # Repair JSON using json-repair (fixes missing commas, quotes, etc.)
    repaired = repair_json(js2)
    return json.loads(repaired)


In [ ]:
JSON_FIXER_PROMPT = """
You returned JSON but it was invalid and could not be parsed.
Fix ONLY the JSON formatting issues (missing commas, unescaped quotes, trailing commas, broken brackets).
Do NOT change the meaning or add/remove test cases.
Return ONLY the corrected JSON. No markdown. No commentary.

INVALID JSON:
\"\"\"{bad_json}\"\"\"
"""

def generate_testcases_from_description(screen_id: str, topic: str, description: str, max_new_tokens=1100):
    prompt = TESTCASE_PROMPT_TEMPLATE.format(
        screen_id=screen_id,
        topic=topic,
        description=description
    )

    raw = generate_text_only(prompt, max_new_tokens=max_new_tokens)

    # 1st attempt: parse directly (with repair)
    try:
        data = parse_testcases_json(raw)
    except Exception:
        # 2nd attempt: ask the model to fix its own JSON
        bad = _extract_json(raw) or raw
        fix_prompt = JSON_FIXER_PROMPT.format(bad_json=bad[:12000])  # safety cap
        fixed_raw = generate_text_only(fix_prompt, max_new_tokens=1200)
        data = parse_testcases_json(fixed_raw)

    # minimal validation
    if "test_cases" not in data or not isinstance(data["test_cases"], list):
        raise ValueError("JSON missing 'test_cases' list.")
    return data


In [ ]:
# ============================================
# CELL 20 — Save per-screen CSV + combined CSV
# ============================================
import os, csv

CSV_COLUMNS = [
    "test_case_id",
    "title",
    "module",
    "screen_id",
    "topic",
    "priority",
    "type",
    "preconditions",
    "test_data",
    "test_steps",
    "expected_result"
]

def write_testcases_csv(screen_id: str, topic: str, module: str, test_cases: list, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{screen_id}.csv")

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        w.writeheader()

        for tc in test_cases:
            w.writerow({
                "test_case_id": tc.get("test_case_id", ""),
                "title": tc.get("title", ""),
                "module": module,
                "screen_id": screen_id,
                "topic": topic,
                "priority": tc.get("priority", ""),
                "type": tc.get("type", ""),
                "preconditions": " | ".join(tc.get("preconditions", []) or []),
                "test_data": " | ".join(tc.get("test_data", []) or []),
                "test_steps": " | ".join(tc.get("steps", []) or []),
                "expected_result": " | ".join(tc.get("expected", []) or []),
            })

    return out_path


def append_to_master_csv(rows: list, master_csv_path: str):
    os.makedirs(os.path.dirname(master_csv_path), exist_ok=True) if os.path.dirname(master_csv_path) else None
    exists = os.path.exists(master_csv_path)

    with open(master_csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not exists:
            w.writeheader()
        for r in rows:
            w.writerow(r)


In [ ]:
def load_description_txt(screen_id: str, in_dir: str = desc_txt_dir):
    p = os.path.join(in_dir, f"{screen_id}.txt")
    if not os.path.exists(p):
        raise FileNotFoundError(f"Description TXT not found: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return f.read().strip()


def stage2_generate_testcases_for_screen(screen_id: str, master_csv_path: str = "/content/testcases_all.csv"):
    # Safe topic lookup (won't crash if csv missing key)
    topic = id_to_topic.get(screen_id, "unknown") if "id_to_topic" in globals() else "unknown"

    desc = load_description_txt(screen_id, in_dir=desc_txt_dir)
    if not desc:
        raise ValueError(f"Empty description txt for screen_id={screen_id}")

    data = generate_testcases_from_description(screen_id, topic, desc, max_new_tokens=1200)

    module = (data.get("module") or "unknown").strip() or "unknown"
    test_cases = data.get("test_cases", [])

    # Write per-screen CSV
    per_csv = write_testcases_csv(screen_id, topic, module, test_cases, out_dir=tc_csv_dir)
    print(f"[OK] Saved per-screen CSV: {per_csv}")

    # Append to master
    rows = []
    for tc in test_cases:
        rows.append({
            "test_case_id": tc.get("test_case_id", ""),
            "title": tc.get("title", ""),
            "module": module,
            "screen_id": screen_id,
            "topic": topic,
            "priority": tc.get("priority", ""),
            "type": tc.get("type", ""),
            "preconditions": " | ".join(tc.get("preconditions", []) or []),
            "test_data": " | ".join(tc.get("test_data", []) or []),
            "test_steps": " | ".join(tc.get("steps", []) or []),
            "expected_result": " | ".join(tc.get("expected", []) or []),
        })

    append_to_master_csv(rows, master_csv_path)
    print(f"[OK] Appended to master CSV: {master_csv_path}")

    return per_csv, master_csv_path


In [ ]:
screen_id = "15660"  # change to 2040 / 124 / anything
per_csv, master_csv = stage2_generate_testcases_for_screen(screen_id)

print("\nPer-screen CSV:", per_csv)
print("Master CSV:", master_csv)


TypeError: sequence item 0: expected str instance, dict found

In [ ]:
!zip -r /content/testcases.zip /content/testcases